# Hybrid dataset — label HotpotQA + CoQA with Qwen2.5-7B, split over several Kaggle accounts

About 3,000 questions (1,500 HotpotQA multi-hop + 1,500 CoQA conversation turns), split over 2 accounts:
each labels ~1,500 (about 1.5 h including set-up).

**Each account (teammates run it in their own account — Kaggle allows one account per person):**
1. The `prob-reas` dataset must be visible to the account: on the dataset page → *Settings → Sharing* → add the
   teammate's Kaggle username (or make it public). Upload the NEW zip first (it contains `build_hybrid.py`).
2. Import this notebook, *Add Input* → `prob-reas`, **GPU T4 x2**, **Internet on**.
3. *Add-ons → Secrets* → `HF_TOKEN` = the SAME Hugging Face write token in every account, so all parts are
   backed up to one private dataset (each part is its own file, nothing overwrites anything).
4. In the first code cell set **`SHARD`**: 0 in your account, 1 in your friend's (`NUM_SHARDS` stays 2).
5. **Save Version → Save & Run All (Commit).**

When all parts are on Hugging Face: download them into `prob_reas/labels/` and run `merge_labels.py`.

In [ ]:
# >>> the ONLY thing to change: SHARD = 0 in your account, SHARD = 1 in your friend's <<<
SHARD = 0
NUM_SHARDS = 2
LLM = 'qwen2.5-7b'
assert 0 <= SHARD < NUM_SHARDS

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, glob, shutil
hits = glob.glob('/kaggle/input/**/build_hybrid.py', recursive=True)
assert hits, 'Add the prob-reas dataset (NEW version with build_hybrid.py) as an input'
shutil.copytree(os.path.dirname(hits[0]), '/kaggle/working/project', dirs_exist_ok=True)
os.chdir('/kaggle/working/project')
os.environ['HF_HOME'] = '/tmp/hf'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print(sorted(os.listdir('.'))[:30])

In [ ]:
# HF token from Kaggle Secrets -> backups every 50 questions. Refuses to run without it.
from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    raise SystemExit('No HF_TOKEN secret attached (Add-ons -> Secrets)') from e
from huggingface_hub import HfApi
print('HF_TOKEN OK, logged in as', HfApi().whoami()['name'])
CHECKPOINT = '--checkpoint auto --checkpoint-every 50'

In [ ]:
!pip install -q spacy click pgmpy faiss-cpu accelerate datasets 2>&1 | grep -v "already satisfied" | tail -3
!python -m spacy download en_core_web_sm -q 2>&1 | tail -2

In [ ]:
# Build the hybrid questions + knowledge base (identical in every account: fixed seed)
!python build_hybrid.py --out kb_hybrid --hotpot 1500 --coqa-stories 250

In [ ]:
# Label this account's share
!python label_dataset.py --llm {LLM} --kb kb_hybrid --limit 0 --out /kaggle/working/labels --num-shards {NUM_SHARDS} --shard {SHARD} {CHECKPOINT}

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/labels', 'zip', '/kaggle/working/labels')
print(os.listdir('/kaggle/working/labels'))